# Введение в MapReduce модель на Python


In [20]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [21]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [22]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [23]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [24]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [25]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [26]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [27]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [146]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [29]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [30]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [31]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [32]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [33]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [34]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.5891979712859574)),
 (1, np.float64(2.5891979712859574)),
 (2, np.float64(2.5891979712859574)),
 (3, np.float64(2.5891979712859574)),
 (4, np.float64(2.5891979712859574))]

## Inverted index

In [35]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [36]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [37]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [38]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('what', 10)]),
 (1, [('a', 2), ('it', 18)])]

## TeraSort

In [39]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.13124872109934538)),
   (None, np.float64(0.14436824019798433)),
   (None, np.float64(0.23722576685546326)),
   (None, np.float64(0.24366397661101402)),
   (None, np.float64(0.2530780647208606)),
   (None, np.float64(0.26042970709578317)),
   (None, np.float64(0.2613304712031843)),
   (None, np.float64(0.3102931358899621)),
   (None, np.float64(0.35540931466601366)),
   (None, np.float64(0.3913642347945898)),
   (None, np.float64(0.4101247664942609)),
   (None, np.float64(0.42034675488926265)),
   (None, np.float64(0.4634047021536518)),
   (None, np.float64(0.4674078642314108)),
   (None, np.float64(0.48031002065279904)),
   (None, np.float64(0.48127907187872))]),
 (1,
  [(None, np.float64(0.5733257518852266)),
   (None, np.float64(0.5744896123526194)),
   (None, np.float64(0.6309718670931392)),
   (None, np.float64(0.6654026802644434)),
   (None, np.float64(0.6655771209631858)),
   (None, np.float64(0.6666911278817762)),
   (None, np.float64(0.7274698858055

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

In [40]:
from typing import Iterator, NamedTuple, Callable, Iterable, Tuple, Any
import random

In [41]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(
        map(lambda x: REDUCE(*x),
            groupbykey(
                flatten(
                    map(lambda x: MAP(*x), RECORDREADER())
                )
            )
        )
    )

In [42]:
count = 20
numbers = [random.randint(0, 100) for _ in range(count)]

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [43]:
def max_value(numbers: list):
    def RECORDREADER():
        for i, n in enumerate(numbers):
            yield (i, n)

    def MAP(key, value):
        yield ("max", value)

    def REDUCE(key, values: Iterator):
        yield (key, max(values))

    return list(MapReduce(RECORDREADER, MAP, REDUCE))

In [44]:
result = max_value(numbers)
print(f"Numbers: {numbers}")
print(f"Max: {result[0][1]}")

Numbers: [45, 45, 94, 71, 46, 45, 76, 36, 37, 30, 57, 59, 0, 6, 75, 2, 27, 58, 28, 80]
Max: 94


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [45]:
def arithmetic_mean(numbers: list):
    def RECORDREADER():
        for i, n in enumerate(numbers):
            yield (i, n)

    def MAP(key, value):
        yield ("mean", (value, 1))

    def REDUCE(key, values: Iterator):
        total, count = 0, 0
        for (v, c) in values:
            total += v
            count += c
        yield (key, total / count)

    return list(MapReduce(RECORDREADER, MAP, REDUCE))

In [46]:
result = arithmetic_mean(numbers)
print(f"Numbers: {numbers}")
print(f"Mean: {result[0][1]}")

Numbers: [45, 45, 94, 71, 46, 45, 76, 36, 37, 30, 57, 59, 0, 6, 75, 2, 27, 58, 28, 80]
Mean: 45.85


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [148]:
def RECORDREADER():
    # возвращаем две map-партиции
    yield [(1,'a'), (2,'b'), (1,'c')]
    yield [(2,'d'), (3,'e'), (1,'f')]

def MAP(k, v):
    # identity map
    yield (k, v)

def REDUCE(key, values):
    # объединяем значения в строку
    yield (key, ','.join(values))

In [149]:
map_outputs = []
for record_reader in RECORDREADER():
    for k,v in record_reader:
        for kv in MAP(k,v):
            map_outputs.append(kv)

grouped = groupbykey(map_outputs)

reduce_outputs = []
for k, vs in grouped:
    for out in REDUCE(k, vs):
        reduce_outputs.append(out)

reduce_outputs

[(1, 'a,c,f'), (2, 'b,d'), (3, 'e')]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [64]:
def drop_duplicates(items: list):
    def INPUTFORMAT():
        global mappers
        chunk_size = (len(items) + mappers - 1)
        for i in range(0, len(items), chunk_size):
            chunk = items[i:i + chunk_size]
            yield [(j, v) for j, v in enumerate(chunk)]

    def MAP(key, value):
        yield (value, 1)

    def REDUCE(key, values):
        yield (key, None)

    def COMBINER(key, values):
        yield (key, 1)

    result = MapReduceDistributed(
        INPUTFORMAT,
        MAP,
        REDUCE,
        COMBINER=COMBINER
    )

    return (list(flatten([v for (_, v) in result])))

In [65]:
mappers = 3
reducers = 2

items = [1, 2, 2, 3, 3, 3, 4, 3, 2, 4, 5, 4, 2, 1, 3, 6]

result = drop_duplicates(items)
print(f"Drop duplicates {items}: {sorted(result)}")

6 key-value pairs were sent over a network.
Drop duplicates [1, 2, 2, 3, 3, 3, 4, 3, 2, 4, 5, 4, 2, 1, 3, 6]: [(1, None), (2, None), (3, None), (4, None), (5, None), (6, None)]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [66]:
class User(NamedTuple):
    name: str
    age: int

In [67]:
R = [User("Alice", 23), User("Bob", 30), User("Charlie", 25)]
S = [User("Bob", 30), User("Diana", 22), User("Alice", 25)]

In [68]:
def selection(relation: list, predicate):
    def RECORDREADER():
        for i, t in enumerate(relation):
            yield (i, t)

    def MAP(key, t):
        if predicate(t):
            yield (t, t)

    def REDUCE(t, values: Iterator):
        _ = list(values)
        yield (t, t)

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [69]:
result = selection(R, predicate=lambda t: t.age >= 25)
print(f"Selection (age >= 25): {result}")

Selection (age >= 25): [User(name='Bob', age=30), User(name='Charlie', age=25)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [70]:
def projection(relation: list, attributes: tuple):
    def project_tuple(t):
        return tuple(getattr(t, a) for a in attributes)

    def RECORDREADER():
        for i, t in enumerate(relation):
            yield (i, t)

    def MAP(key, t):
        t_proj = project_tuple(t)
        yield (t_proj, t_proj)

    def REDUCE(t_proj, values: Iterator):
        _ = list(values)
        yield (t_proj, t_proj)  # дубликаты отброшены группировкой

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [71]:
result = projection(R, attributes=("age",))
print(f"Projection (age only): {result}")

Projection (age only): [(23,), (30,), (25,)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [72]:
def union(relation_R: list, relation_S: list):
    combined = relation_R + relation_S

    def RECORDREADER():
        for i, t in enumerate(combined):
            yield (i, t)

    def MAP(key, t):
        yield (t, t)

    def REDUCE(t, values: Iterator):
        _ = list(values)
        yield (t, t)

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [127]:
result = union(R, S)
print(f"Union R ∪ S: {sorted(result)}")

Union R ∪ S: [User(name='Alice', age=23), User(name='Alice', age=25), User(name='Bob', age=30), User(name='Charlie', age=25), User(name='Diana', age=22)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [124]:
def intersection(relation_R: list, relation_S: list):
    combined = [(t, "R") for t in relation_R] + [(t, "S") for t in relation_S]

    def RECORDREADER():
        for i, item in enumerate(combined):
            yield (i, item)

    def MAP(key, item):
        t, source = item
        yield (t, source)

    def REDUCE(t, sources: Iterator):
        sources_list = list(sources)
        # Кортеж входит в пересечение, только если есть в обоих отношениях
        if "R" in sources_list and "S" in sources_list:
            yield (t, t)

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [126]:
result = intersection(R, S)
print(f"Intersection R ∩ S: {sorted(result)}")

Intersection R ∩ S: [User(name='Bob', age=30)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [76]:
def difference(relation_R: list, relation_S: list):
    combined = [(t, 0) for t in relation_R] + [(t, 1) for t in relation_S]

    def RECORDREADER():
        for i, item in enumerate(combined):
            yield (i, item)

    def MAP(key, item):
        t, source_bit = item
        yield (t, source_bit)

    def REDUCE(t, source_bits: Iterator):
        bits = list(source_bits)
        if all(b == 0 for b in bits):
            yield (t, t)

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [77]:
result = difference(R, S)
print(f"Difference R − S: {result}")

result_inv = difference(S, R)
print(f"Difference S − R: {result_inv}")

Difference R − S: [User(name='Alice', age=23), User(name='Charlie', age=25)]
Difference S − R: [User(name='Diana', age=22), User(name='Alice', age=25)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [78]:
def natural_join(relation_R: list, relation_S: list):
    combined = [(t, "R") for t in relation_R] + [(t, "S") for t in relation_S]

    def RECORDREADER():
        for i, item in enumerate(combined):
            yield (i, item)

    def MAP(key, item):
        t, rel = item
        if rel == "R":
            a, b = t
            yield (b, ("R", a))
        else:
            b, c = t
            yield (b, ("S", c))

    def REDUCE(b, pairs: Iterator):
        pairs_list = list(pairs)
        r_values = [a for (rel, a) in pairs_list if rel == "R"]
        s_values = [c for (rel, c) in pairs_list if rel == "S"]
        for a in r_values:
            for c in s_values:
                yield (None, (a, b, c))

    return [v for (_, v) in MapReduce(RECORDREADER, MAP, REDUCE)]

In [79]:
employees = [("Alice", "Engineering"), ("Bob", "Marketing"), ("Charlie", "Engineering")]
departments = [("Engineering", "Berlin"), ("Marketing", "Paris"), ("HR", "London")]

result = natural_join(employees, departments)
print(f"Natural Join:")
for triple in sorted(result):
    print(f"    {triple}")

Natural Join:
    ('Alice', 'Engineering', 'Berlin')
    ('Bob', 'Marketing', 'Paris')
    ('Charlie', 'Engineering', 'Berlin')


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [80]:
def grouping_and_aggregation(relation: list, theta: Callable):
    def RECORDREADER():
        for i, t in enumerate(relation):
            yield (i, t)

    def MAP(key, t):
        a, b, c = t
        yield (a, b)

    def REDUCE(a, values: Iterator):
        yield (a, theta(list(values)))

    return list(MapReduce(RECORDREADER, MAP, REDUCE))

In [81]:
salaries = [
    ("Engineering", 90000, None),
    ("Engineering", 80000, None),
    ("Marketing",   70000, None),
    ("Marketing",   75000, None),
    ("HR",          60000, None),
]

result_sum = grouping_and_aggregation(salaries, theta=sum)
print(f"Grouping + SUM по зарплате:")
for row in sorted(result_sum):
    print(f"    {row}")

result_max = grouping_and_aggregation(salaries, theta=max)
print(f"Grouping + MAX по зарплате:")
for row in sorted(result_max):
    print(f"    {row}")

Grouping + SUM по зарплате:
    ('Engineering', 170000)
    ('HR', 60000)
    ('Marketing', 145000)
Grouping + MAX по зарплате:
    ('Engineering', 90000)
    ('HR', 60000)
    ('Marketing', 75000)


### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [142]:
def RECORDREADER():
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield ((0, i, j), M[i,j])

    for j in range(V.shape[0]):
        yield ((1, j, 0), V[j])

def MAP_JOIN(k1, v1):
    (typ, a, b) = k1
    val = v1

    if typ == 0:  # matrix
        i = a
        j = b
        yield (j, ('M', i, val))

    else:  # vector
        j = a
        yield (j, ('V', val))

def REDUCE_JOIN(j, values):
    matrix_vals = []
    vector_val = None

    for v in values:
        if v[0] == 'M':
            matrix_vals.append((v[1], v[2]))
        else:
            vector_val = v[1]

    if vector_val is not None:
        for (i, m_ij) in matrix_vals:
            yield (i, m_ij * vector_val)


def JOINED_BY_CHUNK():
    global joined_parts
    for j in joined_parts:
        yield j

def MAP_SUM(k, v):
    yield (k, v)

def REDUCE_SUM(i, values):
    yield (i, sum(values))

In [145]:
import numpy as np

I = 5
J = 7

M = np.random.rand(I, J)
V = np.random.rand(J)

reference_solution = M @ V

joined_parts = MapReduce(RECORDREADER, MAP_JOIN, REDUCE_JOIN)
solution = MapReduce(JOINED_BY_CHUNK, MAP_SUM, REDUCE_SUM)
result = np.zeros(I)

for (i, val) in solution:
    result[i] = val

np.allclose(reference_solution, result)

True

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [114]:
import numpy as np

I = 2
J = 3
K = 4 * 10

small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1

    for i in range(small_mat.shape[0]):
        v = small_mat[i, j]
        yield ((i, k), v * w)

def REDUCE(key, values):
    (i, k) = key
    yield ((i, k), sum(values))

Проверьте своё решение

In [115]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i,k), vw) in reduce_output)+1
    K = max(k for ((i,k), vw) in reduce_output)+1
    mat = np.empty(shape=(I,K))
    for ((i,k), vw) in reduce_output:
        mat[i,k] = vw
    return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [116]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

Перемножение матриц через два шага MapReduce:
1. JOIN по индексу j
2. SUM по (i,k)

In [134]:
def RECORDREADER():
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            yield ((0, i, j), small_mat[i,j])

    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((1, j, k), big_mat[j,k])

def MAP_JOIN(k1, v1):
    (mat_num, i, j) = k1
    w = v1
    if mat_num == 0:
        yield (j, (mat_num, i, w))
    else:
        yield (i, (mat_num, j, w))

def REDUCE_JOIN(key, values):
    from_first_mat = [v for v in values if v[0] == 0]
    from_second_mat = [v for v in values if v[0] == 1]
    for f in from_first_mat:
        for s in from_second_mat:
            yield ((f[1], s[1]), f[2] * s[2])



def JOINED_BY_CHUNK():
    global joined_parts
    for j in joined_parts:
        yield j

def MAP_MUL(k1, v1):
    (i, k) = k1
    yield ((i, k), v1)

def REDUCE_MUL(key, values):
    res_el_value = 0
    for v in values:
        res_el_value += v
    yield (key, res_el_value)


In [135]:
I = 2
J = 3
K = 4 * 10

small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

reference_solution = np.matmul(small_mat, big_mat)


joined_parts = MapReduce(RECORDREADER, MAP_JOIN, REDUCE_JOIN)
solution = MapReduce(JOINED_BY_CHUNK, MAP_MUL, REDUCE_MUL)
np.allclose(reference_solution, asmatrix(solution))

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [119]:
def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
    map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
    if COMBINER != None:
        map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)

    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)
    reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
    return reduce_outputs

In [136]:
def INPUTFORMAT():
    first_mat = []
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            first_mat.append(((0, i, j), small_mat[i,j]))
    yield first_mat

    second_mat = []
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            second_mat.append(((1, j, k), big_mat[j,k]))
    yield second_mat

def MAP_JOIN(k1, v1):
    (mat_num, i, j) = k1
    w = v1
    if mat_num == 0:
        yield (j, (mat_num, i, w))
    else:
        yield (i, (mat_num, j, w))

def REDUCE_JOIN(key, values):
    from_first_mat = [v for v in values if v[0] == 0]
    from_second_mat = [v for v in values if v[0] == 1]
    for f in from_first_mat:
        for s in from_second_mat:
            yield ((f[1], s[1]), f[2] * s[2])


def JOINED_BY_CHUNK():
    global joined_parts
    for j in joined_parts:
        yield j[1]

def MAP_MUL(k1, v1):
    yield (k1, v1)

def REDUCE_MUL(key, values):
    res_val = 0
    for v in values:
        res_val += v
    yield (key, res_val)

In [137]:
maps = 4
reducers = 2

partition_res = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN, COMBINER=None)
joined_parts = [(partition_id, list(partition)) for (partition_id, partition) in partition_res]

mul_res = MapReduceDistributed(JOINED_BY_CHUNK, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_res]

solution = [v for p in pre_result for v in p[1]]
np.allclose(reference_solution, asmatrix(solution))

True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [138]:
def INPUTFORMAT():
    global maps
    first_mat = []
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            first_mat.append(((0, i, j), small_mat[i,j]))
    split_size = int(np.ceil(len(first_mat)/maps))
    for i in range(0, len(first_mat), split_size):
        yield first_mat[i:i+split_size]

    second_mat = []
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            second_mat.append(((1, j, k), big_mat[j,k]))
    split_size = int(np.ceil(len(second_mat)/maps))
    for i in range(0, len(second_mat), split_size):
        yield second_mat[i:i+split_size]

def MAP_JOIN(k1, v1):
    (mat_num, i, j) = k1
    w = v1

    if mat_num == 0:
        yield (j, (mat_num, i, w))
    else:
        yield (i, (mat_num, j, w))

def REDUCE_JOIN(key, values):
    from_first_mat = [v for v in values if v[0] == 0]
    from_second_mat = [v for v in values if v[0] == 1]

    for f in from_first_mat:
        for s in from_second_mat:
            yield ((f[1], s[1]), f[2] * s[2])

def JOINED_BY_CHUNK():
    global joined_parts
    for j in joined_parts:
        yield j[1]

def MAP_MUL(k1, v1):
    yield (k1, v1)

def REDUCE_MUL(key, values):
    res_val = 0
    for v in values:
        res_val += v
    yield (key, res_val)

In [139]:
maps = 3
reducers = 2

partition_res = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN, COMBINER=None)
joined_parts = [(partition_id, list(partition)) for (partition_id, partition) in partition_res]

mul_res = MapReduceDistributed(JOINED_BY_CHUNK, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_res]

solution = [v for p in pre_result for v in p[1]]
np.allclose(reference_solution, asmatrix(solution))

True